# Drought Event — SSI Temporal Evolution
**Target journal:** Computers and Geosciences  
**Purpose:** Two-panel figure showing the daily SSI evolution of a single drought event at any gauging station.  
- **Panel (a):** Daily SSI time series with severity-tiered filled areas, secondary log-discharge axis, event statistics box, seasonal background shading, and context window.  
- **Panel (b):** SSI distribution histogram with KDE and WMO drought severity threshold lines.  

**Usage:** Set `STATION_ID`, `EVENT_ID`, and `WINDOW` in Cell 4, then run all cells.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 – Imports & global rcParams
# ─────────────────────────────────────────────────────────────────────────────
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
from scipy.stats import gaussian_kde

mpl.rcParams.update({
    'font.family'       : 'serif',
    'font.serif'        : ['Times New Roman', 'DejaVu Serif', 'serif'],
    'font.size'         : 8,
    'axes.linewidth'    : 0.7,
    'xtick.major.width' : 0.7,
    'ytick.major.width' : 0.7,
    'xtick.minor.width' : 0.4,
    'ytick.minor.width' : 0.4,
    'xtick.direction'   : 'out',
    'ytick.direction'   : 'out',
    'figure.dpi'        : 150,
    'savefig.dpi'       : 300,
})

print('Libraries loaded.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 – File paths
# ─────────────────────────────────────────────────────────────────────────────
SSI_DAILY  = os.path.join('data', 'SSI_daily.csv')
EVENTS     = os.path.join('data', 'SSI_drought_events.csv')
STATIONS   = os.path.join('data', 'station_locations_final_FILTRADO.csv')
OUTPUT_DIR = os.path.join('output', 'figures')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 – Load data  (run once; reuse for different events)
# ─────────────────────────────────────────────────────────────────────────────
ssi_df      = pd.read_csv(SSI_DAILY,  parse_dates=['date'])
events_df   = pd.read_csv(EVENTS,     parse_dates=['start_date', 'end_date', 'peak_date'])
stations_df = pd.read_csv(STATIONS)

print(f'SSI records    : {len(ssi_df):,}')
print(f'Drought events : {len(events_df):,}')
print(f'Stations       : {len(stations_df)}')
print(f'SSI columns    : {ssi_df.columns.tolist()}')
print(f'Event columns  : {events_df.columns.tolist()}')

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#  CELL 4 – USER PARAMETERS  ← edit here
# ═════════════════════════════════════════════════════════════════════════════
STATION_ID = 9011   # integer station code
EVENT_ID   = 12     # event number for that station
WINDOW     = 5     # context days before and after the event
# ═════════════════════════════════════════════════════════════════════════════

# Quick reference: list all events for the selected station
avail = (
    events_df[events_df['station_id'] == STATION_ID]
    [['event_id', 'start_date', 'end_date', 'duration', 'severity', 'intensity', 'peak_SSI']]
    .reset_index(drop=True)
)
print(f'Events available for station {STATION_ID}:\n')
print(avail.to_string())

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 – Extract event metadata & build time window
# ─────────────────────────────────────────────────────────────────────────────

# ── Event row ─────────────────────────────────────────────────────────────────
mask = (events_df['station_id'] == STATION_ID) & (events_df['event_id'] == EVENT_ID)
if not mask.any():
    valid = sorted(events_df[events_df['station_id'] == STATION_ID]['event_id'].tolist())
    raise ValueError(f'Event {EVENT_ID} not found for station {STATION_ID}. Valid IDs: {valid}')

ev       = events_df[mask].iloc[0]
ev_start = ev['start_date']
ev_end   = ev['end_date']
ev_peak  = ev['peak_date']
duration = int(ev['duration'])
severity = float(ev['severity'])
intensity= float(ev['intensity'])
peak_ssi = float(ev['peak_SSI'])

# Volumetric deficit column (handle naming variants)
vol_col = next((c for c in events_df.columns if 'hm' in c.lower()), None)
sev_vol = float(ev[vol_col]) if vol_col else None

# ── Station metadata ──────────────────────────────────────────────────────────
st_row       = stations_df[stations_df['station_id'] == STATION_ID]
station_name = st_row['station_name'].iloc[0]  if not st_row.empty else str(STATION_ID)
st_regime    = st_row['régimen'].iloc[0]        if not st_row.empty else 'unknown'

# ── SSI window ────────────────────────────────────────────────────────────────
win_start = ev_start - pd.Timedelta(days=WINDOW)
win_end   = ev_end   + pd.Timedelta(days=WINDOW)

ssi_win = (
    ssi_df[ssi_df['station_id'] == STATION_ID]
    .query('@win_start <= date <= @win_end')
    .copy()
)
if ssi_win.empty:
    raise ValueError(f'No SSI data for station {STATION_ID} in [{win_start.date()} – {win_end.date()}]')

dates_pd = pd.to_datetime(ssi_win['date'].values)
ssi      = ssi_win['SSI'].values.astype(float)
Q        = ssi_win['Q'].values.astype(float)

print(f'Station  : {STATION_ID} — {station_name}  [{st_regime}]')
print(f'Event    : {EVENT_ID}   {ev_start.date()} → {ev_end.date()}   ({duration} d)')
print(f'Severity : {severity:.2f}  |  Intensity: {intensity:.4f} d⁻¹  |  Peak SSI: {peak_ssi:.2f}')
if sev_vol is not None:
    print(f'Vol. deficit: {sev_vol:.1f} hm³')
print(f'Window   : {win_start.date()} → {win_end.date()}  ({len(ssi_win)} days)')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 – Constants & helper functions
# ─────────────────────────────────────────────────────────────────────────────

# WMO drought severity thresholds (standard normal percentiles)
T_MOD = -1.28   # P10  – moderate drought onset
T_SEV = -1.65   # P05  – severe drought
T_EXT = -2.00   # P02  – extreme drought

# Severity fill colours
C_MOD = '#fed976'   # amber
C_SEV = '#fd8d3c'   # orange
C_EXT = '#bd0026'   # dark red

# Seasonal background colours (very light, non-intrusive)
SEASON_COLS = {
    'DJF': '#ddeeff',   # winter  – pale blue
    'MAM': '#eef7e8',   # spring  – pale green
    'JJA': '#fffae8',   # summer  – pale yellow
    'SON': '#f7ede8',   # autumn  – pale orange
}

def _season(month):
    if month in (12, 1, 2): return 'DJF'
    if month in (3, 4, 5):  return 'MAM'
    if month in (6, 7, 8):  return 'JJA'
    return 'SON'


def add_seasonal_background(ax, dates_pd):
    """
    Draw alternating seasonal background bands on ax.
    Returns list of (season_label, mid_date) tuples for annotation.
    """
    seasons = [_season(d.month) for d in dates_pd]
    spans   = []
    i = 0
    while i < len(dates_pd):
        s = seasons[i]
        j = i + 1
        while j < len(dates_pd) and seasons[j] == s:
            j += 1
        right = (
            dates_pd[j] if j < len(dates_pd)
            else dates_pd[-1] + pd.Timedelta(days=1)
        )
        mid = dates_pd[i] + (right - dates_pd[i]) / 2
        ax.axvspan(
            dates_pd[i], right,
            facecolor=SEASON_COLS[s], alpha=0.50,
            zorder=0, label='_nolegend_'
        )
        spans.append((s, mid))
        i = j
    return spans


print('Constants and helpers defined.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 – Build & save figure
# ─────────────────────────────────────────────────────────────────────────────

FIG_W = 7.48   # 190 mm – double-column width
FIG_H = 6.30   # 160 mm

fig = plt.figure(figsize=(FIG_W, FIG_H))
gs  = gridspec.GridSpec(
    2, 1, figure=fig,
    height_ratios=[3, 1.2],
    hspace=0.36,
    left=0.09, right=0.87, top=0.92, bottom=0.09,
)
ax1 = fig.add_subplot(gs[0])   # SSI time series
ax2 = fig.add_subplot(gs[1])   # histogram

# ══════════════════════════════════════════════════════════════════════════════
# PANEL (a) — SSI time series
# ══════════════════════════════════════════════════════════════════════════════


# (ii) Context window shading (grey overlay on pre/post-event days)
ax1.axvspan(
    dates_pd[0], ev_start,
    facecolor='#999999', alpha=0.12, zorder=1, label='Context window'
)
ax1.axvspan(
    ev_end, dates_pd[-1],
    facecolor='#999999', alpha=0.12, zorder=1, label='_nolegend_'
)

# (iii) Reference lines
ax1.axhline(0,     color='#888888', lw=0.55, ls='-',  zorder=2)
ax1.axhline(T_MOD, color='#333333', lw=1.00, ls='--', zorder=3,
            label=f'Threshold (SSI = {T_MOD})')
ax1.axhline(T_SEV, color=C_SEV,    lw=0.70, ls=':',  zorder=3)
ax1.axhline(T_EXT, color=C_EXT,    lw=0.70, ls=':',  zorder=3)

# (iv) Severity-tiered filled areas  (layered: amber → orange → red)
kw_fill = dict(interpolate=True, zorder=4)
ax1.fill_between(dates_pd, ssi, T_MOD, where=ssi <= T_MOD,
                 fc=C_MOD, alpha=0.85, label='Moderate drought', **kw_fill)
ax1.fill_between(dates_pd, ssi, T_SEV, where=ssi <= T_SEV,
                 fc=C_SEV, alpha=0.85, label='Severe drought',   **kw_fill)
ax1.fill_between(dates_pd, ssi, T_EXT, where=ssi <= T_EXT,
                 fc=C_EXT, alpha=0.90, label='Extreme drought',  **kw_fill)

# (v) Daily SSI line
ax1.plot(dates_pd, ssi, color='#2166ac', lw=1.0, zorder=7, label='Daily SSI')

# (vi) Event boundary vertical lines
ax1.axvline(ev_start, color='#333333', lw=0.80, ls='-', alpha=0.50, zorder=8)
ax1.axvline(ev_end,   color='#333333', lw=0.80, ls='-', alpha=0.50, zorder=8)

# (vii) Peak SSI marker
ax1.axvline(
    ev_peak, color='#7b2d8b', lw=0.90, ls='--', alpha=0.80, zorder=8,
    label=f'Peak ({ev_peak.strftime("%d %b %Y")})',
)
ax1.scatter(
    [ev_peak], [peak_ssi],
    color='#7b2d8b', s=35, marker='v',
    edgecolors='white', linewidths=0.5, zorder=9,
)

# (viii) Secondary discharge axis — log scale
ax1q = ax1.twinx()
Q_safe = np.where(Q > 0, Q, np.nan)
ax1q.plot(dates_pd, Q_safe, color='#525252', lw=0.65, alpha=0.38, zorder=3)
ax1q.set_yscale('log')
ax1q.set_ylabel(r'$Q$ (m³ s⁻¹)', fontsize=7.5, fontfamily='serif',
                color='#525252', labelpad=5)
ax1q.tick_params(axis='y', labelsize=6.5, colors='#525252')
ax1q.spines['right'].set_linewidth(0.6)
ax1q.spines['right'].set_edgecolor('#525252')
for sp in ['top', 'bottom', 'left']:
    ax1q.spines[sp].set_visible(False)

# (ix) Event statistics annotation box
vol_line = f'\nVol. deficit : {sev_vol:.1f} hm³' if sev_vol is not None else ''
stats_text = (
    f'Duration   : {duration} d\n'
    f'Severity   : {severity:.2f}\n'
    f'Intensity  : {intensity:.4f} d⁻¹\n'
    f'Peak SSI   : {peak_ssi:.2f}\n'
    f'Peak date  : {ev_peak.strftime("%d %b %Y")}'
    + vol_line
)
ax1.text(
    0.040, 0.975, stats_text,
    transform=ax1.transAxes,
    fontsize=6.2, fontfamily='serif',
    va='top', linespacing=1.55,
    bbox=dict(boxstyle='round,pad=0.45', fc='white',
              ec='#888888', lw=0.6, alpha=0.92),
    zorder=10,
)

# (x) Axes limits, ticks, labels
ax1.set_xlim(dates_pd[0], dates_pd[-1])
y_lo = min(np.nanmin(ssi) - 0.5, T_EXT - 0.5)
y_hi = max(np.nanmax(ssi) + 0.5, 1.8)
ax1.set_ylim(y_lo, y_hi)
ax1.set_ylabel('SSI (–)', fontsize=8, fontfamily='serif')
ax1.yaxis.set_major_locator(mticker.MultipleLocator(1.0))
ax1.yaxis.set_minor_locator(mticker.MultipleLocator(0.5))
ax1.xaxis.set_major_locator(mdates.DayLocator(interval=7))
ax1.xaxis.set_minor_locator(mdates.DayLocator(interval=1))
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
ax1.tick_params(axis='x', labelsize=7, rotation=30)
ax1.tick_params(axis='y', labelsize=7)

# (xi) Legend — deduplicate handles
h_all, l_all = ax1.get_legend_handles_labels()
seen_l, h_leg, l_leg = set(), [], []
for hh, ll in zip(h_all, l_all):
    if ll not in seen_l and not ll.startswith('_'):
        seen_l.add(ll); h_leg.append(hh); l_leg.append(ll)
ax1.legend(
    h_leg, l_leg,
    loc='upper right', fontsize=6, ncol=2,
    framealpha=0.88, edgecolor='black', fancybox=False,
    borderpad=0.5, handlelength=1.4, columnspacing=0.8,
)

# (xii) Panel label
ax1.text(0.005, 0.975, '(a)', transform=ax1.transAxes,
         fontsize=9, fontweight='bold', fontfamily='serif', va='top')

# ══════════════════════════════════════════════════════════════════════════════
# PANEL (b) — SSI distribution: histogram + KDE + severity thresholds
# ══════════════════════════════════════════════════════════════════════════════
event_mask = (dates_pd >= ev_start) & (dates_pd <= ev_end)
ssi_event  = ssi[event_mask]
ssi_ok     = ssi_event[np.isfinite(ssi_event)]

# Histogram with severity-coloured bars
n_bins = min(40, max(15, len(ssi_ok) // 8))
_, bins, patches = ax2.hist(
    ssi_ok, bins=n_bins, density=True,
    edgecolor='white', linewidth=0.3, zorder=3,
)
for patch, left in zip(patches, bins[:-1]):
    if   left < T_EXT: patch.set_facecolor(C_EXT); patch.set_alpha(0.80)
    elif left < T_SEV: patch.set_facecolor(C_SEV); patch.set_alpha(0.80)
    elif left < T_MOD: patch.set_facecolor(C_MOD); patch.set_alpha(0.80)
    else:              patch.set_facecolor('#6baed6'); patch.set_alpha(0.60)

# KDE curve
if len(ssi_ok) >= 10:
    kde  = gaussian_kde(ssi_ok, bw_method='silverman')
    xk   = np.linspace(ssi_ok.min() - 0.5, ssi_ok.max() + 0.5, 500)
    ax2.plot(xk, kde(xk), color='#08306b', lw=1.4, zorder=5, label='KDE')

# Threshold vertical lines and severity labels
# get_xaxis_transform(): x in data coordinates, y in axes fraction
tier_info = [
    (T_MOD, '#333333', 'Moderate (−1.28)'),
    (T_SEV, C_SEV,     'Severe (−1.65)'),
    (T_EXT, C_EXT,     'Extreme (−2.00)'),
]
trans = ax2.get_xaxis_transform()
for xv, col, lbl in tier_info:
    ax2.axvline(xv, color=col, lw=0.9, ls='--', zorder=4)
    ax2.text(
        xv - 0.06, 0.96, lbl,
        transform=trans, ha='right', va='top',
        fontsize=5.8, fontfamily='serif', color=col,
        rotation=90,
    )

# Axes
ax2.set_xlabel('SSI (–)', fontsize=8, fontfamily='serif')
ax2.set_ylabel('Density', fontsize=8, fontfamily='serif')
ax2.tick_params(axis='both', labelsize=7)
ax2.xaxis.set_major_locator(mticker.MultipleLocator(1.0))
ax2.xaxis.set_minor_locator(mticker.MultipleLocator(0.5))
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
_leg_handles = [
    Patch(facecolor='#6baed6', alpha=0.60, label='Near normal'),
    Patch(facecolor=C_MOD,     alpha=0.80, label='Moderate drought'),
    Patch(facecolor=C_SEV,     alpha=0.80, label='Severe drought'),
    Patch(facecolor=C_EXT,     alpha=0.90, label='Extreme drought'),
]
if len(ssi_ok) >= 10:
    _leg_handles.append(Line2D([0], [0], color='#08306b', lw=1.4, label='KDE'))
ax2.legend(
    handles=_leg_handles,
    fontsize=6.5, framealpha=0.88, edgecolor='black',
    fancybox=False, borderpad=0.5, handlelength=1.4,
)
ax2.text(0.005, 0.975, '(b)', transform=ax2.transAxes,
         fontsize=9, fontweight='bold', fontfamily='serif', va='top')

# ── Figure title ──────────────────────────────────────────────────────────────
fig.suptitle(
    f'Station {STATION_ID} — {station_name}  ·  Event {EVENT_ID}  '
    f'({ev_start.strftime("%d %b %Y")} – {ev_end.strftime("%d %b %Y")})  ',
    fontsize=8.5, fontfamily='serif', y=0.975,
)

# ── Save ──────────────────────────────────────────────────────────────────────
fname = f'drought_event_S{STATION_ID}_E{EVENT_ID}.png'
out   = os.path.join(OUTPUT_DIR, fname)
fig.savefig(out, dpi=300, bbox_inches='tight', pad_inches=0.05,
            facecolor='white', transparent=False)
print(f'Figure saved → {out}')

plt.show()